# AMANAH v0.2 — Export the already-trained checkpoint (CPU only)

Use this notebook in **free Colab CPU**. It does not train or evaluate. It only mounts Google Drive, finds the completed v0.2 checkpoint/cache, packages it with the current GitHub code, and uploads a private **`v0.2-candidate`** branch to the same Hugging Face model repository.

This preserves the training already completed on Colab so Kaggle can continue evaluation without retraining.


In [ ]:
# 1) Mount Drive and locate the completed AMANAH cache
from google.colab import drive, userdata
from pathlib import Path
import json, os, shutil, subprocess, sys
drive.mount('/content/drive', force_remount=False)
CACHE_ROOT=Path('/content/drive/MyDrive/semantic_integrity_model_cache')
runs=sorted((CACHE_ROOT/'runs').glob('*'), key=lambda p:p.stat().st_mtime, reverse=True)
candidates=[]
for run in runs:
    ck=run/'checkpoint'
    if (ck/'training_complete.json').exists() and (ck/'model.pt').exists() and (ck/'amanah_config.json').exists():
        candidates.append(run)
assert candidates, f'No completed checkpoint found under {CACHE_ROOT}/runs'
RUN_CACHE=candidates[0]
checkpoint_dir=RUN_CACHE/'checkpoint'
dataset_dir=RUN_CACHE/'semantic_dataset'
source_store=CACHE_ROOT/'sources'/'reference_store.json'
assert source_store.exists(), source_store
assert (dataset_dir/'validation.jsonl').exists(), dataset_dir
assert (dataset_dir/'test.jsonl').exists(), dataset_dir
print('Using run cache:',RUN_CACHE)
print('Checkpoint:',checkpoint_dir)


In [ ]:
# 2) Clone the current GitHub v0.2 branch and package the checkpoint
repo_dir=Path('/content/Amanah')
if repo_dir.exists(): shutil.rmtree(repo_dir)
subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/Saahar2001/Amanah-.git',str(repo_dir)],check=True)
project_dir=repo_dir/'Amanah AI model'
assert project_dir.exists(), project_dir
os.chdir(project_dir)
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.35,<2'],check=True)
package_dir=Path('/content/amanah_v02_candidate')
subprocess.run([sys.executable,'-m','scripts.package_hf_model','--checkpoint',str(checkpoint_dir),'--reference-store',str(source_store),'--output',str(package_dir)],check=True)
eval_dir=package_dir/'eval'; eval_dir.mkdir(exist_ok=True)
for name in ['validation.jsonl','test.jsonl','dataset_summary.json','qa_report.json']:
    src=dataset_dir/name
    if src.exists(): shutil.copy2(src,eval_dir/name)
print('Packaged files:',sorted(str(p.relative_to(package_dir)) for p in package_dir.rglob('*') if p.is_file())[:40])


In [ ]:
# 3) Upload to the SAME private Hugging Face model repo on a candidate branch
from huggingface_hub import HfApi, create_repo
hf_token=userdata.get('HF_TOKEN')
if not hf_token: raise RuntimeError('Add HF_TOKEN to Colab Secrets. WRITE permission is required.')
api=HfApi(token=hf_token)
identity=api.whoami(); namespace=identity.get('name')
assert namespace
MODEL_REPO=f'{namespace}/semantic-integrity-v0-1'
create_repo(MODEL_REPO,repo_type='model',private=True,exist_ok=True,token=hf_token)
api.create_branch(repo_id=MODEL_REPO,branch='v0.2-candidate',repo_type='model',exist_ok=True,token=hf_token)
info=api.upload_folder(repo_id=MODEL_REPO,repo_type='model',folder_path=str(package_dir),revision='v0.2-candidate',token=hf_token,commit_message='Upload AMANAH v0.2 candidate checkpoint and exact held-out split')
print('UPLOAD COMPLETE')
print('Model repo:',f'https://huggingface.co/{MODEL_REPO}/tree/v0.2-candidate')
print('Candidate commit:',info.oid if hasattr(info,'oid') else info)
print('No production endpoint was changed.')
